# Notebook MLP — Réseau Électrique

**Ce notebook est la version MLP du notebook GNN spatial électrique.**

### Ce qui est conservé
- Mêmes fichiers d'entrée/sortie, mêmes features, mêmes cibles (`p_bus_generator` — 15 bus générateurs × 24h).
- Même historique (72h), même horizon (24h), même stride (24h), même split train/val/test.
- Même normalisation (`StandardScaler`), même logique de fenêtrage, même évaluation (MAE, RMSE, NMAE, persistence, diagnostics).
- Même optimizer (Adam), même scheduler (ReduceLROnPlateau), même early stopping, même seed.

### Ce qui change
- **Suppression de `torch_geometric`** : plus de `Data`, `DataLoader` PyG, `GINEConv`, `NNConv`, `edge_index`, `edge_attr`.
- **Architecture** : le GNN est remplacé par un MLP fully connected (`input_dim → 512 → 256 → 128 → output_dim`).
- **Flattening** : l'entrée `x` de forme `(num_nodes, expected_x_dim)` est aplatie en un vecteur 1D `(num_nodes × expected_x_dim,)` par fenêtre — le MLP voit l'intégralité du graphe en un seul vecteur et peut apprendre des dépendances croisées inter-nœuds.
- **Dataset/DataLoader** : `torch.utils.data.TensorDataset` et `DataLoader` standard ; plus de batching PyG.
- **Sortie** : `(batch_size, N_GEN, HOR)` → reshape interne, compatible avec toutes les fonctions d'évaluation d'origine.


In [ ]:
# Cellule 0 — Installation
# MODIFIÉE : suppression de torch_geometric, plus nécessaire pour le MLP.
import sys, subprocess, pkgutil

def ensure_package(pkg_name, pip_name=None):
    if pkgutil.find_loader(pkg_name) is None:
        name = pip_name or pkg_name
        print(f"Installing {name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])
    else:
        print(f"{pkg_name} already installed.")

# torch_geometric N'EST PLUS NÉCESSAIRE pour ce notebook MLP.
for pkg, pip in [("torch", "torch"), ("sklearn", "scikit-learn")]:
    ensure_package(pkg, pip)


In [ ]:
# Cellule 1 — Configuration
# MODIFIÉE : clés GNN (MODEL_TYPE, NUM_MP_LAYERS, RESIDUAL, ENCODER_HIDDEN_DIMS, DECODER_HIDDEN_DIMS)
# conservées pour compatibilité mais interpretées comme paramètres MLP.
# Ajout de MLP_HIDDEN_DIMS.
from pathlib import Path

use_day_embedding = True
day_emb_dim       = 3

CONFIG = {
    "BASE_DIR"      : Path("."),
    "PATH_NETWORK"  : Path("./Elec Network Data.xlsx"),
    "PATH_WIND"     : Path("./input/Wind Data.xlsx"),
    "PATH_LOAD"     : Path("./input/Elec Load Data.xlsx"),
    "PATH_DECISION" : Path("./output/Decision_variablesRU.xlsx"),

    "MONTHS"   : [f"{m:02d}" for m in range(1, 13)],
    "NUM_NODES": 45,
    "GENERATOR_BUS_IDX": [6, 12, 14, 15, 18, 26, 27, 28, 32, 35, 37, 39, 41, 43, 44],
    "NUM_GENERATORS": 24,

    "HIST_HOURS"    : 72,
    "HORIZON_HOURS" : 24,
    "STRIDE_HOURS"  : 24,

    "TRAIN_END": "2024-10-01 00:00:00",
    "VAL_END"  : "2024-11-01 00:00:00",

    "USE_LOAD_HISTORY"         : True,
    "USE_WIND_OFFSHORE_HISTORY": True,
    "USE_WIND_ONSHORE_HISTORY" : True,
    "USE_P_HISTORY"            : False,

    "USE_NODE_STATIC_IS_GENERATOR"   : True,
    "USE_NODE_STATIC_GENERATOR_COUNT": True,
    "USE_NODE_STATIC_PMAX_TOTAL"     : True,
    "USE_NODE_STATIC_LOAD_SHARE"     : True,
    "USE_NODE_STATIC_PV_SHARE"       : True,
    "USE_NODE_STATIC_ONSHORE_SHARE"  : True,
    "USE_NODE_STATIC_IS_OFFSHORE"    : True,
    "USE_NODE_STATIC_IS_IMPORT"      : True,
    "USE_NODE_STATIC_IMPORT_PMAX"    : True,
    "USE_NODE_STATIC_VOLTAGE_KV"     : True,
    "USE_NODE_STATIC_IS_TRANSFER"    : True,
    "USE_NODE_STATIC_DEGREE"         : True,

    # EDGE cols conservés pour référence mais non utilisés par le MLP
    "EDGE_STATIC_COLUMNS": ["X_pu", "b_pu", "Pmax_MW", "Length_km", "Inom_A", "UB_kV"],
    "USE_BIDIRECTIONAL_EDGES": True,

    # Paramètres MLP (remplacent le GNN)
    "MLP_HIDDEN_DIMS" : [512, 256, 128],  # couches cachées du MLP
    "HIDDEN_DIM"      : 128,              # conservé pour compatibilité (utilisé comme dernier hidden)
    "ENCODER_HIDDEN_DIMS" : [256, 128],   # conservé pour compatibilité
    "DECODER_HIDDEN_DIMS" : [128, 64],    # conservé pour compatibilité
    "DROPOUT"             : 0.15,
    "USE_BATCHNORM"       : True,

    "SEED"                    : 42,
    "DEVICE"                  : "cpu",
    "EPOCHS"                  : 200,
    "BATCH_SIZE"              : 24,
    "LR"                      : 7e-4,
    "WEIGHT_DECAY"            : 1e-4,
    "GRAD_CLIP_NORM"          : 2.0,
    "EARLY_STOPPING_PATIENCE" : 30,
    "EARLY_STOPPING_MIN_DELTA": 1e-4,
    "SCHEDULER_FACTOR"        : 0.4,
    "SCHEDULER_PATIENCE"      : 6,
    "SCHEDULER_MIN_LR"        : 1e-6,

    "PERSISTENCE_MODE"     : "last_24h",
    "NUM_ALIGNMENT_CHECKS" : 5,
    "SAVE_MODEL_STATE"     : False,
    "MODEL_STATE_PATH"     : Path("mlp_elec.pt"),
}

PATHS = {
    "network" : CONFIG["BASE_DIR"] / CONFIG["PATH_NETWORK"],
    "wind"    : CONFIG["BASE_DIR"] / CONFIG["PATH_WIND"],
    "load"    : CONFIG["BASE_DIR"] / CONFIG["PATH_LOAD"],
    "decision": CONFIG["BASE_DIR"] / CONFIG["PATH_DECISION"],
}

TRAIN_END = CONFIG["TRAIN_END"]
VAL_END   = CONFIG["VAL_END"]
HIST      = CONFIG["HIST_HOURS"]
HOR       = CONFIG["HORIZON_HOURS"]
STRIDE    = CONFIG["STRIDE_HOURS"]
N_GEN     = len(CONFIG["GENERATOR_BUS_IDX"])

print("Fichiers attendus :")
for k, v in PATHS.items():
    print(f"  {k}: {v.resolve()} | exists={v.exists()}")
for p in PATHS.values():
    assert p.exists(), f"Fichier introuvable: {p}"
print("\nCONFIG OK.")
print(f"use_day_embedding={use_day_embedding}  day_emb_dim={day_emb_dim}")


In [ ]:
# Cellule 2 — Imports
# MODIFIÉE : suppression de tous les imports torch_geometric.
# Utilisation du DataLoader standard de PyTorch.
import copy
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Pas de from torch_geometric ...

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["SEED"])

DEVICE = torch.device(
    "cuda" if (CONFIG["DEVICE"] == "cuda" and torch.cuda.is_available()) else "cpu"
)
print("DEVICE =", DEVICE)

def pretty_shape(name, x):
    if isinstance(x, (np.ndarray, torch.Tensor, pd.DataFrame, pd.Series)):
        print(f"{name}: shape={getattr(x, 'shape', None)}")
    else:
        print(f"{name}: type={type(x)}")

def drop_unnamed_first_col(df):
    if len(df.columns) > 0 and str(df.columns[0]).startswith("Unnamed"):
        return df.drop(columns=[df.columns[0]])
    return df

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Chargement des données réseau (cellules identiques au notebook GNN)

In [ ]:
# Cellule 3 — Chargement des fichiers réseau
# IDENTIQUE au notebook GNN.
buses_df   = pd.read_excel(PATHS["network"], sheet_name="Buses")
lines_df   = pd.read_excel(PATHS["network"], sheet_name="LinesMerged")
gen_df     = pd.read_excel(PATHS["network"], sheet_name="Gen")
genmap_df  = pd.read_excel(PATHS["network"], sheet_name="GenMap",  header=0)
windmap_df = pd.read_excel(PATHS["network"], sheet_name="WindMap", header=0)
impmap_df  = pd.read_excel(PATHS["network"], sheet_name="ImpMap",  header=0)
imp_df     = pd.read_excel(PATHS["network"], sheet_name="Imp")
renew_df   = pd.read_excel(PATHS["network"], sheet_name="Renew")

num_nodes         = CONFIG["NUM_NODES"]
generator_bus_idx = CONFIG["GENERATOR_BUS_IDX"]

for name, df in [("buses_df", buses_df), ("lines_df", lines_df),
                 ("gen_df", gen_df), ("genmap_df", genmap_df),
                 ("windmap_df", windmap_df), ("impmap_df", impmap_df)]:
    pretty_shape(name, df)

assert len(buses_df)  == num_nodes
assert len(gen_df)    == CONFIG["NUM_GENERATORS"]
assert genmap_df.shape == (num_nodes, CONFIG["NUM_GENERATORS"])
assert windmap_df.shape[0] == num_nodes

print("\nColonnes Buses :", list(buses_df.columns))
print("Colonnes Lines :", list(lines_df.columns))
print("Colonnes Gen   :", list(gen_df.columns))
display(buses_df.head())
display(lines_df.head())
display(gen_df)


In [ ]:
# Cellule 4 — Mapping générateurs → bus
# IDENTIQUE au notebook GNN.
genmap = genmap_df.to_numpy(dtype=float)

generator_to_bus = {}
for gen_idx in range(CONFIG["NUM_GENERATORS"]):
    connected = np.where(genmap[:, gen_idx] > 0.5)[0]
    assert len(connected) == 1, f"Gen {gen_idx} connecté à {len(connected)} bus"
    generator_to_bus[gen_idx] = int(connected[0])

print("Mapping gen → bus (0-based):")
for g, b in generator_to_bus.items():
    print(f"  Gen{g+1:2d} → Bus {b+1:2d} ({buses_df.iloc[b]['Name']})")

pmax_per_bus = np.zeros(num_nodes, dtype=float)
for gen_idx, bus_idx in generator_to_bus.items():
    pmax_val = gen_df.iloc[gen_idx]["Pmax_MW"]
    if isinstance(pmax_val, str):
        pmax_val = eval(pmax_val.replace("=", ""))
    pmax_per_bus[bus_idx] += float(pmax_val)

gen_count_per_bus = np.zeros(num_nodes, dtype=int)
for gen_idx, bus_idx in generator_to_bus.items():
    gen_count_per_bus[bus_idx] += 1

gen_buses_from_map = sorted(set(generator_to_bus.values()))
assert gen_buses_from_map == sorted(CONFIG["GENERATOR_BUS_IDX"])
print(f"\n{len(gen_buses_from_map)} bus générateurs (0-based): {gen_buses_from_map}")

windmap        = windmap_df.to_numpy(dtype=float).flatten()
offshore_bus_idx = np.where(windmap > 0.5)[0]
assert len(offshore_bus_idx) == 1 and offshore_bus_idx[0] == 38
print(f"Bus offshore : bus {offshore_bus_idx[0]} (STEVN), windmap OK")

impmap = impmap_df.to_numpy(dtype=float)
is_import_bus  = (impmap.sum(axis=1) > 0).astype(float)
import_pmax_mw = np.zeros(num_nodes, dtype=float)
for i in range(num_nodes):
    for j in range(8):
        if impmap[i, j] > 0.5:
            import_pmax_mw[i] += float(imp_df.iloc[j]["Pmax_MW"])
print(f"Bus d'interconnexion (1-based): {np.where(is_import_bus > 0)[0] + 1}")


In [ ]:
# Cellule 5 — Shares PV / onshore / load
# IDENTIQUE au notebook GNN.
transfos   = buses_df["Transfos"].to_numpy(dtype=float)
load_share = transfos / transfos.sum()
assert np.isclose(load_share.sum(), 1.0)
print("load_share.sum() =", load_share.sum())

PV_SHARES_RAW = {
    1: 3.455/2,  2: 2.192/4,  4: 16.02/2,  5: 7.093/11,
    7: 15.46/3,  8: 7.093/11, 10: 3.455/2, 13: 17.37/4,
   14: 17.37/4, 16: 7.625/2, 17: 7.093/11, 19: 16.02/2,
   20: 7.093/11, 21: 17.26/4, 22: 7.093/11, 23: 7.093/11,
   25: 2.192/4, 26: 17.26/4, 27: 17.26/4,  28: 15.46/3,
   29: 17.37/4, 31: 7.093/11, 32: 7.093/11, 33: 17.37/4,
   34: 7.093/11, 35: 2.192/4, 36: 7.093/11, 38: 7.625/2,
   40: 7.093/11, 42: 15.46/3, 43: 2.192/4, 44: 17.26/4,
   45: 13.52,
}
ONSHORE_SHARES_RAW = {
    1: 11.599/2,  2: 3.459/4,  4: 8.741/2,  5: 9.825/11,
    7: 5.15/3,   8: 9.825/11, 10: 11.599/2, 13: 18.252/4,
   14: 18.252/4, 16: 17.785/2, 17: 9.825/11, 19: 8.741/2,
   20: 9.825/11, 21: 14.835/4, 22: 9.825/11, 23: 9.825/11,
   25: 3.459/4,  26: 14.835/4, 27: 14.835/4, 28: 5.15/3,
   29: 18.252/4, 31: 9.825/11, 32: 9.825/11, 33: 18.252/4,
   34: 9.825/11, 35: 3.459/4,  36: 9.825/11, 38: 17.785/2,
   40: 9.825/11, 42: 5.15/3,  43: 3.459/4,  44: 14.835/4,
   45: 10.35,
}

pv_share      = np.array([PV_SHARES_RAW.get(i+1, 0.0)      for i in range(num_nodes)], dtype=float)
onshore_share = np.array([ONSHORE_SHARES_RAW.get(i+1, 0.0) for i in range(num_nodes)], dtype=float)

pv_share_norm      = pv_share / (pv_share.sum() + 1e-12)
onshore_share_norm = onshore_share / (onshore_share.sum() + 1e-12)
print("pv_share.sum()      =", pv_share.sum())
print("onshore_share.sum() =", onshore_share.sum())


In [ ]:
# Cellule 6 — Fonctions de lecture des fichiers
# IDENTIQUE au notebook GNN.
def read_decision_p(months, decision_path):
    all_arrays = []
    for m in months:
        df = pd.read_excel(decision_path, sheet_name=f"p-{m}-2024", header=0, index_col=0)
        all_arrays.append(df.to_numpy(dtype=float))
    p_full = np.concatenate(all_arrays, axis=0)
    ts_p   = pd.date_range("2024-01-01", periods=len(p_full), freq="h")
    return p_full, ts_p

def read_elec_load(months, load_path):
    dfs = []
    for m in months:
        df = pd.read_excel(load_path, sheet_name=f"{m}-2024", header=0, usecols=[0, 1])
        df.columns = ["datetime", "load_MW"]
        df["datetime"] = pd.to_datetime(df["datetime"])
        dfs.append(df[["datetime", "load_MW"]])
    df_all = pd.concat(dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
    return df_all["load_MW"].to_numpy(dtype=float), pd.DatetimeIndex(df_all["datetime"])

def read_wind(months, wind_path):
    dfs = []
    for m in months:
        df = pd.read_excel(wind_path, sheet_name=f"{m}-2024", header=0, usecols=[0, 1])
        df.columns = ["datetime", "wind_MW"]
        df["datetime"] = pd.to_datetime(df["datetime"])
        dfs.append(df[["datetime", "wind_MW"]])
    df_all = pd.concat(dfs, ignore_index=True).sort_values("datetime").reset_index(drop=True)
    return df_all["wind_MW"].to_numpy(dtype=float), pd.DatetimeIndex(df_all["datetime"])


In [ ]:
# Cellule 7 — Chargement des séries temporelles
# IDENTIQUE au notebook GNN.
p_full,     ts_p    = read_decision_p(CONFIG["MONTHS"], PATHS["decision"])
load_total, ts_load = read_elec_load(CONFIG["MONTHS"],  PATHS["load"])
wind_total, ts_wind = read_wind(CONFIG["MONTHS"],       PATHS["wind"])

assert len(ts_p) == len(ts_load) == len(ts_wind)
assert p_full.shape[1] == 24

T = len(ts_p)
print("Plage temporelle :", ts_p.min(), "->", ts_p.max())
print("Nombre total d'heures :", T)

pretty_shape("p_full [T, 24]",   p_full)
pretty_shape("load_total [T]",   load_total)
pretty_shape("wind_total [T]",   wind_total)


In [ ]:
# Cellule 8 — Construction des séries nodales
# IDENTIQUE au notebook GNN.
p_bus_full = np.zeros((T, num_nodes), dtype=float)
for gen_idx, bus_idx in generator_to_bus.items():
    p_bus_full[:, bus_idx] += p_full[:, gen_idx]

p_bus_generator = p_bus_full[:, CONFIG["GENERATOR_BUS_IDX"]]

np.testing.assert_allclose(p_bus_full.sum(axis=1), p_full.sum(axis=1), rtol=1e-6)
print("Agrégation p par bus OK.")

load_nodal          = load_total[:, None] * load_share[None, :]
wind_offshore_nodal = np.zeros((T, num_nodes), dtype=float)
wind_offshore_nodal[:, 38] = wind_total
wind_onshore_nodal  = wind_total[:, None] * onshore_share_norm[None, :]
p_history_nodal     = p_bus_full.copy()

pretty_shape("p_bus_generator [T, 15]",    p_bus_generator)
pretty_shape("load_nodal [T, 45]",         load_nodal)
pretty_shape("wind_offshore_nodal [T, 45]",wind_offshore_nodal)


In [ ]:
# Cellule 9 — Construction du tenseur dynamique
# IDENTIQUE au notebook GNN.
dyn_feature_dict = {}

if CONFIG["USE_LOAD_HISTORY"]:
    dyn_feature_dict["load_nodal"] = load_nodal
if CONFIG["USE_WIND_OFFSHORE_HISTORY"]:
    dyn_feature_dict["wind_offshore"] = wind_offshore_nodal
if CONFIG["USE_WIND_ONSHORE_HISTORY"]:
    dyn_feature_dict["wind_onshore"] = wind_onshore_nodal
if CONFIG["USE_P_HISTORY"]:
    dyn_feature_dict["p_history"] = p_history_nodal

assert len(dyn_feature_dict) > 0, "Au moins une feature dynamique doit être activée."

dyn_feature_names = list(dyn_feature_dict.keys())
dyn_tensor        = np.stack([dyn_feature_dict[k] for k in dyn_feature_names], axis=-1)

print("Features dynamiques :", dyn_feature_names)
pretty_shape("dyn_tensor [T, N, F_dyn]", dyn_tensor)

display(pd.DataFrame({
    "feature": dyn_feature_names,
    "shape"  : [str(dyn_feature_dict[k].shape) for k in dyn_feature_names],
    "min"    : [dyn_feature_dict[k].min()  for k in dyn_feature_names],
    "max"    : [dyn_feature_dict[k].max()  for k in dyn_feature_names],
    "mean"   : [dyn_feature_dict[k].mean() for k in dyn_feature_names],
}))


In [ ]:
# Cellule 10 — Features statiques nodales
# IDENTIQUE au notebook GNN.
degrees = np.zeros(num_nodes, dtype=float)
for _, row in lines_df.iterrows():
    u = int(row["FromNode"]) - 1
    v = int(row["ToNode"])   - 1
    degrees[u] += 1
    degrees[v] += 1

is_transfer = (buses_df["Type"] == "Transfer").to_numpy(dtype=float)
voltage_kv  = buses_df["UB_kV"].to_numpy(dtype=float)

node_static_parts = []
node_static_names = []

_ns = [
    ("USE_NODE_STATIC_IS_GENERATOR",    lambda: (gen_count_per_bus > 0).astype(float)[:, None],          "is_generator"),
    ("USE_NODE_STATIC_GENERATOR_COUNT", lambda: gen_count_per_bus[:, None].astype(float),                "generator_count"),
    ("USE_NODE_STATIC_PMAX_TOTAL",      lambda: pmax_per_bus[:, None],                                   "pmax_total_MW"),
    ("USE_NODE_STATIC_LOAD_SHARE",      lambda: load_share[:, None],                                     "load_share"),
    ("USE_NODE_STATIC_PV_SHARE",        lambda: pv_share_norm[:, None],                                  "pv_share"),
    ("USE_NODE_STATIC_ONSHORE_SHARE",   lambda: onshore_share_norm[:, None],                             "onshore_share"),
    ("USE_NODE_STATIC_IS_OFFSHORE",     lambda: windmap[:, None],                                        "is_offshore_wind"),
    ("USE_NODE_STATIC_IS_IMPORT",       lambda: is_import_bus[:, None],                                  "is_import_bus"),
    ("USE_NODE_STATIC_IMPORT_PMAX",     lambda: import_pmax_mw[:, None],                                 "import_pmax_MW"),
    ("USE_NODE_STATIC_VOLTAGE_KV",      lambda: voltage_kv[:, None],                                     "voltage_kV"),
    ("USE_NODE_STATIC_IS_TRANSFER",     lambda: is_transfer[:, None],                                    "is_transfer"),
    ("USE_NODE_STATIC_DEGREE",          lambda: degrees[:, None],                                        "degree"),
]

for key, builder, name in _ns:
    if CONFIG.get(key, False):
        node_static_parts.append(builder())
        node_static_names.append(name)

node_static = (
    np.concatenate(node_static_parts, axis=1)
    if node_static_parts
    else np.zeros((num_nodes, 0), dtype=float)
)

pretty_shape("node_static [N, F_static]", node_static)
print("node_static_names =", node_static_names)


In [ ]:
# Cellule 11 — Construction de edge_index / edge_attr (conservée mais non utilisée par le MLP)
# NOTE MLP : cette cellule est conservée pour compatibilité et référence, mais le MLP n'utilise
# ni edge_index ni edge_attr. On la garde pour documenter la topologie du réseau.
edge_static_cols = CONFIG["EDGE_STATIC_COLUMNS"]
edge_static_base = (
    lines_df[edge_static_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .to_numpy(dtype=float)
)

edge_pairs = []
edge_attrs = []
for i, row in lines_df.iterrows():
    u  = int(row["FromNode"]) - 1
    v  = int(row["ToNode"])   - 1
    ea = edge_static_base[i]
    edge_pairs.append((u, v))
    edge_attrs.append(ea)
    if CONFIG["USE_BIDIRECTIONAL_EDGES"]:
        edge_pairs.append((v, u))
        edge_attrs.append(ea.copy())

# Gardés comme arrays NumPy (pas de conversion en tenseur PyG)
edge_pairs_arr = np.array(edge_pairs, dtype=np.int64)   # (E, 2) — pour référence uniquement
edge_attr_raw  = np.array(edge_attrs, dtype=float)       # (E, F_edge) — pour référence uniquement

print("edge_static_cols =", edge_static_cols)
print("Nb arêtes :", len(edge_pairs_arr))
pretty_shape("edge_pairs_arr (non utilisé par le MLP)", edge_pairs_arr)
pretty_shape("edge_attr_raw  (non utilisé par le MLP)", edge_attr_raw)


In [ ]:
# Cellule 12 — Jour de semaine / week-end
# IDENTIQUE au notebook GNN.
ts_p = pd.to_datetime(ts_p)

day_of_week = ts_p.dayofweek.to_numpy(dtype=np.int64)
is_weekend  = (day_of_week >= 5).astype(np.int64)

print("Plage temporelle       :", ts_p.min(), "->", ts_p.max())
print("day_of_week [0=Mon … 6=Sun]  min/max:", day_of_week.min(), "/", day_of_week.max())
print("is_weekend (0/1)             sum:", is_weekend.sum(), "heures week-end")

pd.DataFrame({
    "datetime"   : ts_p[:8],
    "day_of_week": day_of_week[:8],
    "is_weekend" : is_weekend[:8],
}).pipe(display)


In [ ]:
# Cellule 13 — Normalisation et split temporel
# IDENTIQUE au notebook GNN.
TRAIN_END_ts = pd.Timestamp(TRAIN_END)
VAL_END_ts   = pd.Timestamp(VAL_END)

train_time_mask = ts_p < TRAIN_END_ts
val_time_mask   = (ts_p >= TRAIN_END_ts) & (ts_p < VAL_END_ts)
test_time_mask  = ts_p >= VAL_END_ts

print("Train hours :", int(train_time_mask.sum()))
print("Val hours   :", int(val_time_mask.sum()))
print("Test hours  :", int(test_time_mask.sum()))

T, N_nodes, F_dyn = dyn_tensor.shape

dyn_scaler = StandardScaler()
dyn_scaler.fit(dyn_tensor[train_time_mask].reshape(-1, F_dyn))
dyn_tensor_scaled = dyn_scaler.transform(dyn_tensor.reshape(-1, F_dyn)).reshape(T, N_nodes, F_dyn)

node_static_scaler = StandardScaler()
node_static_scaler.fit(node_static)
node_static_scaled = node_static_scaler.transform(node_static)

target_scaler = StandardScaler()
target_scaler.fit(p_bus_generator[train_time_mask])
p_bus_gen_scaled = target_scaler.transform(p_bus_generator)

print("\nTarget scaler mean shape :", target_scaler.mean_.shape)
print("Target scaler scale shape:", target_scaler.scale_.shape)
pretty_shape("dyn_tensor_scaled    [T, N_nodes, F_dyn]", dyn_tensor_scaled)
pretty_shape("node_static_scaled   [N_nodes, F_static]", node_static_scaled)
pretty_shape("p_bus_gen_scaled     [T, N_GEN]",          p_bus_gen_scaled)
print("\nNormalisation OK.")


In [ ]:
# Cellule 14 — Fenêtrage temporel + build_single_window
# MODIFIÉE : ajout de flat_input_dim pour le MLP.
# build_single_window retourne maintenant x_flat (1D) en plus de x (2D par nœud).
all_start_indices = np.arange(0, T - HIST - HOR + 1, STRIDE)

forecast_starts   = pd.DatetimeIndex([ts_p[i + HIST] for i in all_start_indices])
train_window_mask = forecast_starts < TRAIN_END_ts
val_window_mask   = (forecast_starts >= TRAIN_END_ts) & (forecast_starts < VAL_END_ts)
test_window_mask  = forecast_starts >= VAL_END_ts

print(f"Nombre total de fenêtres : {len(all_start_indices)}")
print(f"  Train : {int(train_window_mask.sum())}")
print(f"  Val   : {int(val_window_mask.sum())}")
print(f"  Test  : {int(test_window_mask.sum())}")

# Dimension par nœud (identique au GNN)
expected_x_dim = HIST * F_dyn + node_static_scaled.shape[1]
# Dimension du vecteur aplati (input MLP) = num_nodes × expected_x_dim
flat_input_dim = num_nodes * expected_x_dim
print(f"\nexpected_x_dim (par nœud, sans embedding) = {expected_x_dim}")
print(f"flat_input_dim (vecteur MLP, sans embedding) = {flat_input_dim}")
print(f"N_GEN = {N_GEN}")

def build_single_window(start_idx):
    hist_dyn = dyn_tensor_scaled[start_idx : start_idx + HIST]
    hist_dyn = np.transpose(hist_dyn, (1, 0, 2)).reshape(num_nodes, -1)
    x        = np.concatenate([hist_dyn, node_static_scaled], axis=1)  # (num_nodes, expected_x_dim)
    x_flat   = x.reshape(-1)                                            # (num_nodes * expected_x_dim,) — NOUVEAU

    y      = p_bus_gen_scaled[start_idx + HIST : start_idx + HIST + HOR].T
    y_real = p_bus_generator[start_idx + HIST  : start_idx + HIST + HOR].T

    dow_idx = int(day_of_week[start_idx + HIST])

    return {
        "x"                : x.astype(np.float32),       # (num_nodes, expected_x_dim) — conservé
        "x_flat"           : x_flat.astype(np.float32),  # (flat_input_dim,) — NOUVEAU pour MLP
        "y"                : y.astype(np.float32),        # (N_GEN, HOR)
        "y_real"           : y_real.astype(np.float32),   # (N_GEN, HOR)
        "dow"              : dow_idx,
        "start_idx"        : np.int64(start_idx),
        "target_start_idx" : np.int64(start_idx + HIST),
        "target_end_idx"   : np.int64(start_idx + HIST + HOR - 1),
    }

sample0 = build_single_window(int(all_start_indices[0]))
assert sample0["x"].shape      == (num_nodes, expected_x_dim)
assert sample0["x_flat"].shape == (flat_input_dim,)
assert sample0["y"].shape      == (N_GEN, HOR)
print("\nbuild_single_window OK (x et x_flat).")

def check_alignment(start_idx, verbose=True):
    s = build_single_window(start_idx)
    ts0, ts1 = int(s["target_start_idx"]), int(s["target_end_idx"])
    expected = p_bus_generator[ts0:ts1+1].T
    assert np.allclose(expected, s["y_real"]), "Problème d'alignement y_real"
    if verbose:
        print(f"Alignment OK | start={start_idx} | {ts_p[ts0]} → {ts_p[ts1]}")

for idx in [all_start_indices[0], all_start_indices[10], all_start_indices[-1]]:
    check_alignment(int(idx))


In [ ]:
# Cellule 15 — Dataset MLP
# REMPLACÉE : ElecMLPDataset utilise torch.utils.data.Dataset standard
# (plus de torch_geometric.data.Data, plus de DataLoader PyG).
# x_flat est le vecteur d'entrée aplati ; y est la cible (N_GEN, HOR).

class ElecMLPDataset(Dataset):
    def __init__(self, start_indices):
        self.start_indices = np.array(start_indices, dtype=int)

    def __len__(self):
        return len(self.start_indices)

    def __getitem__(self, idx):
        s = build_single_window(int(self.start_indices[idx]))
        return (
            torch.tensor(s["x_flat"], dtype=torch.float32),   # x_flat : (flat_input_dim,)
            torch.tensor(s["y"],      dtype=torch.float32),   # y      : (N_GEN, HOR)
            torch.tensor(s["dow"],    dtype=torch.long),       # dow    : scalaire
            torch.tensor(s["target_start_idx"], dtype=torch.long),  # méta
            torch.tensor(s["y_real"], dtype=torch.float32),   # y_real : (N_GEN, HOR)
        )

train_ds = ElecMLPDataset(all_start_indices[train_window_mask])
val_ds   = ElecMLPDataset(all_start_indices[val_window_mask])
test_ds  = ElecMLPDataset(all_start_indices[test_window_mask])

# DataLoader standard PyTorch (pas PyG)
train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["BATCH_SIZE"], shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

print(f"len(train_ds) = {len(train_ds)}")
print(f"len(val_ds)   = {len(val_ds)}")
print(f"len(test_ds)  = {len(test_ds)}")

x_batch, y_batch, dow_batch, meta_batch, yreal_batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  x_flat : {x_batch.shape}   (batch_size × flat_input_dim)")
print(f"  y      : {y_batch.shape}   (batch_size × N_GEN × HOR)")
print(f"  dow    : {dow_batch.shape}")
print(f"  y_real : {yreal_batch.shape}")
assert x_batch.shape[1]   == flat_input_dim
assert y_batch.shape[1:] == (N_GEN, HOR)
print("Dataset MLP OK.")


In [ ]:
# Cellule 16 — Classe MLP utilitaire
# IDENTIQUE au notebook GNN (déjà présente).
class MLP(nn.Module):
    def __init__(self, dims, dropout=0.0, use_batchnorm=False, activate_last=False):
        super().__init__()
        self.dims         = list(dims)
        self.in_channels  = self.dims[0]
        self.out_channels = self.dims[-1]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            is_last = (i == len(dims) - 2)
            if (not is_last) or activate_last:
                if use_batchnorm:
                    layers.append(nn.BatchNorm1d(dims[i+1]))
                layers.append(nn.ReLU())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:
# Cellule 17 — Architecture ElecMLP
# REMPLACÉE : le GNN spatial est remplacé par un MLP fully connected.
# Principe du flattening :
#   - Entrée GNN : x de forme (num_nodes, expected_x_dim) + passage de messages entre nœuds
#   - Entrée MLP : vecteur aplati (num_nodes × expected_x_dim,) + embedding jour de semaine optionnel
#   - Le MLP voit l'intégralité de l'état nodal en un seul vecteur et apprend des
#     dépendances croisées inter-nœuds sans structure graphe.
# Architecture : flat_input_dim → 512 → 256 → 128 → N_GEN × HOR

class ElecMLP(nn.Module):
    def __init__(
        self,
        flat_input_dim,
        n_gen,
        horizon,
        hidden_dims=(512, 256, 128),
        dropout=0.15,
        use_batchnorm=True,
        use_day_embedding=False,
        day_emb_dim=3,
    ):
        super().__init__()
        self.n_gen             = n_gen
        self.horizon           = horizon
        self.use_day_embedding = use_day_embedding

        if use_day_embedding:
            self.day_embedding = nn.Embedding(7, day_emb_dim)
            in_dim = flat_input_dim + day_emb_dim
        else:
            in_dim = flat_input_dim

        dims = [in_dim] + list(hidden_dims) + [n_gen * horizon]
        self.net = MLP(dims, dropout=dropout, use_batchnorm=use_batchnorm, activate_last=False)

    def forward(self, x_flat, dow=None):
        # x_flat : (batch_size, flat_input_dim)
        if self.use_day_embedding and dow is not None:
            dow_emb = self.day_embedding(dow)            # (batch_size, day_emb_dim)
            x_flat  = torch.cat([x_flat, dow_emb], dim=1)
        out = self.net(x_flat)                           # (batch_size, n_gen * horizon)
        return out.view(-1, self.n_gen, self.horizon)    # (batch_size, N_GEN, HOR)


# Test rapide
model = ElecMLP(
    flat_input_dim    = flat_input_dim,
    n_gen             = N_GEN,
    horizon           = HOR,
    hidden_dims       = CONFIG["MLP_HIDDEN_DIMS"],
    dropout           = CONFIG["DROPOUT"],
    use_batchnorm     = CONFIG["USE_BATCHNORM"],
    use_day_embedding = use_day_embedding,
    day_emb_dim       = day_emb_dim,
).to(DEVICE)

print(model)
print(f"\nParamètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

x_test, y_test, dow_test, _, _ = next(iter(train_loader))
x_test   = x_test.to(DEVICE)
dow_test = dow_test.to(DEVICE)
with torch.no_grad():
    pred_test = model(x_test, dow=dow_test if use_day_embedding else None)
print(f"Forward OK  pred_test.shape = {pred_test.shape}  (attendu: {y_test.shape})")
assert pred_test.shape == y_test.shape


In [ ]:
# Cellule 18 — Fonctions de perte, inverse transform, persistence, run_one_epoch
# MODIFIÉE : adaptation au batch MLP (tuples de tenseurs au lieu de batch PyG).
# La logique de loss, inverse transform et persistence est conservée à l'identique.

def mse_loss_elec(pred, y_true):
    # pred   : (batch_size, N_GEN, HOR)
    # y_true : (batch_size, N_GEN, HOR)
    return F.mse_loss(pred, y_true.view_as(pred))

def inverse_targets_flexible(arr_scaled, n_gen=N_GEN, horizon=HOR):
    arr = np.asarray(arr_scaled)
    scaler_shape = np.asarray(target_scaler.mean_).shape

    if scaler_shape in [(1,), ()]:
        return target_scaler.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)

    if len(scaler_shape) == 1 and scaler_shape[0] == n_gen:
        if arr.ndim == 2:
            bsz  = arr.shape[0] // n_gen
            arr2 = arr.reshape(bsz, n_gen, horizon).transpose(0, 2, 1).reshape(-1, n_gen)
            return target_scaler.inverse_transform(arr2).reshape(bsz, horizon, n_gen).transpose(0, 2, 1).reshape(arr.shape)
        if arr.ndim == 3:
            bsz  = arr.shape[0]
            arr2 = arr.transpose(0, 2, 1).reshape(-1, n_gen)
            return target_scaler.inverse_transform(arr2).reshape(bsz, horizon, n_gen).transpose(0, 2, 1)

    raise ValueError(f"target_scaler incompatible: mean_.shape={scaler_shape}")

def mae_rmse_nmae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).reshape(-1), np.asarray(y_pred).reshape(-1)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    nmae = mae / (np.mean(np.abs(y_true)) + 1e-12)
    return {"MAE": mae, "RMSE": rmse, "NMAE": nmae}

def persistence_forecast_for_sample(start_idx, mode=None):
    mode   = mode or CONFIG["PERSISTENCE_MODE"]
    hist_p = p_bus_generator[start_idx : start_idx + HIST]
    if mode == "last_24h":
        return hist_p[-HOR:].T
    if mode == "last_value":
        return np.repeat(hist_p[[-1]], HOR, axis=0).T
    raise ValueError(f"PERSISTENCE_MODE inconnu: {mode}")

def run_one_epoch(mdl, loader, optimizer=None):
    is_train = optimizer is not None
    mdl.train() if is_train else mdl.eval()
    total_loss, total_samples = 0.0, 0
    for x_flat, y_true, dow, _, _ in loader:
        x_flat  = x_flat.to(DEVICE)
        y_true  = y_true.to(DEVICE)
        dow     = dow.to(DEVICE)
        bsz     = x_flat.shape[0]
        with torch.set_grad_enabled(is_train):
            pred = mdl(x_flat, dow=dow if use_day_embedding else None)
            loss = mse_loss_elec(pred, y_true)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                if CONFIG["GRAD_CLIP_NORM"] is not None:
                    torch.nn.utils.clip_grad_norm_(mdl.parameters(), CONFIG["GRAD_CLIP_NORM"])
                optimizer.step()
        total_loss    += loss.item() * bsz
        total_samples += bsz
    return total_loss / max(total_samples, 1)


In [ ]:
# Cellule 19 — EarlyStopping
# IDENTIQUE au notebook GNN.
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_loss   = np.inf
        self.best_state  = None
        self.counter     = 0
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = float(val_loss)
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


In [ ]:
# Cellule 20 — Instanciation du modèle + optimizer + scheduler
# MODIFIÉE : instanciation de ElecMLP au lieu de SpatialElecGNN.
model = ElecMLP(
    flat_input_dim    = flat_input_dim,
    n_gen             = N_GEN,
    horizon           = HOR,
    hidden_dims       = CONFIG["MLP_HIDDEN_DIMS"],
    dropout           = CONFIG["DROPOUT"],
    use_batchnorm     = CONFIG["USE_BATCHNORM"],
    use_day_embedding = use_day_embedding,
    day_emb_dim       = day_emb_dim,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"]
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=CONFIG["SCHEDULER_FACTOR"],
    patience=CONFIG["SCHEDULER_PATIENCE"], min_lr=CONFIG["SCHEDULER_MIN_LR"],
)
early_stopper = EarlyStopping(
    patience  = CONFIG["EARLY_STOPPING_PATIENCE"],
    min_delta = CONFIG["EARLY_STOPPING_MIN_DELTA"],
)
history = {"train_loss": [], "val_loss": [], "lr": []}

print(model)
print(f"\nParamètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("\nModèle / optimizer / scheduler / history réinitialisés.")


In [ ]:
# Cellule 21 — Boucle d'entraînement
# IDENTIQUE au notebook GNN (run_one_epoch adapté en cell 18).
for epoch in range(1, CONFIG["EPOCHS"] + 1):
    train_loss = run_one_epoch(model, train_loader, optimizer=optimizer)
    val_loss   = run_one_epoch(model, val_loader,   optimizer=None)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)

    print(
        f"Epoch {epoch:03d} | "
        f"train={train_loss:.6f} | val={val_loss:.6f} | "
        f"counter={early_stopper.counter}/{early_stopper.patience} | lr={current_lr:.2e}"
    )

    if early_stopper.step(val_loss, model):
        print(f"Early stopping déclenché à l'epoch {epoch}.")
        break

assert early_stopper.best_state is not None
model.load_state_dict(early_stopper.best_state)
print("\nMeilleur val_loss:", early_stopper.best_loss)

if CONFIG["SAVE_MODEL_STATE"]:
    torch.save(model.state_dict(), CONFIG["MODEL_STATE_PATH"])
    print("Modèle sauvegardé dans", CONFIG["MODEL_STATE_PATH"])


In [ ]:
# Cellule 22 — Courbes d'entraînement
# IDENTIQUE au notebook GNN.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
epochs_ran = range(1, len(history["train_loss"]) + 1)
ax1.plot(epochs_ran, history["train_loss"], label="train")
ax1.plot(epochs_ran, history["val_loss"],   label="val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE Loss")
ax1.set_title("Courbes de perte — ElecMLP"); ax1.legend(); ax1.grid(True)

ax2.plot(epochs_ran, history["lr"])
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Learning rate")
ax2.set_title("Learning rate schedule"); ax2.grid(True)

plt.tight_layout()
plt.show()
print(f"Entraînement terminé en {len(history['train_loss'])} epochs.")
print(f"Best val_loss = {early_stopper.best_loss:.6f}")


In [ ]:
# Cellule 23 — Fonctions utilitaires + collect_predictions
# MODIFIÉE : collect_predictions adapté au DataLoader MLP (tuple de tenseurs).
# ensure_3d_output, inverse_targets_flexible : identiques.

def ensure_3d_output(arr, n_gen, horizon):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        assert arr.shape[1] == n_gen and arr.shape[2] == horizon
        return arr
    if arr.ndim == 2:
        assert arr.shape[1] == horizon
        assert arr.shape[0] % n_gen == 0
        return arr.reshape(arr.shape[0] // n_gen, n_gen, horizon)
    raise ValueError(f"ndim non supporté: {arr.ndim}")

@torch.no_grad()
def collect_predictions(mdl, loader):
    mdl.eval()
    preds_scaled, trues_scaled, preds_real, trues_real, metas = [], [], [], [], []
    for x_flat, y_true, dow, meta, y_real_batch in loader:
        x_flat  = x_flat.to(DEVICE)
        dow     = dow.to(DEVICE)
        pred    = mdl(x_flat, dow=dow if use_day_embedding else None)  # (bsz, N_GEN, HOR)

        pred_np  = pred.detach().cpu().numpy()
        true_np  = y_true.numpy()            # (bsz, N_GEN, HOR)

        preds_scaled.append(pred_np)
        trues_scaled.append(true_np)
        preds_real.append(inverse_targets_flexible(pred_np))
        trues_real.append(inverse_targets_flexible(true_np))
        metas.append(meta.numpy())

    return {
        "preds_scaled": np.concatenate(preds_scaled, axis=0),
        "trues_scaled": np.concatenate(trues_scaled, axis=0),
        "preds_real"  : np.concatenate(preds_real,   axis=0),
        "trues_real"  : np.concatenate(trues_real,   axis=0),
        "metas"       : np.concatenate(metas, axis=0),
    }

train_out = collect_predictions(model, train_loader)
val_out   = collect_predictions(model, val_loader)
test_out  = collect_predictions(model, test_loader)

for split, out in [("train", train_out), ("val", val_out), ("test", test_out)]:
    print(f"{split} | preds_real={out['preds_real'].shape} | trues_real={out['trues_real'].shape}")


In [ ]:
# Cellule 24 — Calcul de la baseline persistence
# IDENTIQUE au notebook GNN.
def compute_persistence_split(dataset, mode=None):
    mode = mode or CONFIG["PERSISTENCE_MODE"]
    preds, trues = [], []
    for start_idx in dataset.start_indices:
        start_idx    = int(start_idx)
        target_start = start_idx + HIST
        pred = persistence_forecast_for_sample(start_idx, mode=mode)
        true = p_bus_generator[target_start : target_start + HOR].T
        preds.append(pred)
        trues.append(true)
    return np.stack(preds, axis=0), np.stack(trues, axis=0)

pers_train_pred, pers_train_true = compute_persistence_split(train_ds)
pers_val_pred,   pers_val_true   = compute_persistence_split(val_ds)
pers_test_pred,  pers_test_true  = compute_persistence_split(test_ds)

print("pers_train_pred.shape =", pers_train_pred.shape)
print("pers_val_pred.shape   =", pers_val_pred.shape)
print("pers_test_pred.shape  =", pers_test_pred.shape)


In [ ]:
# Cellule 25 — Évaluation globale MAE / RMSE / NMAE
# MODIFIÉE : labels "ElecMLP" au lieu de "SpatialElecGNN".
def evaluate_output_dict(out, split_name):
    preds_3d = ensure_3d_output(out["preds_real"], N_GEN, HOR)
    trues_3d = ensure_3d_output(out["trues_real"], N_GEN, HOR)
    rows = []
    m = mae_rmse_nmae(trues_3d, preds_3d)
    m.update({"split": split_name, "model": "ElecMLP_global"})
    rows.append(m)
    for j, bus_idx in enumerate(CONFIG["GENERATOR_BUS_IDX"]):
        m = mae_rmse_nmae(trues_3d[:, j, :], preds_3d[:, j, :])
        m.update({"split": split_name, "model": f"ElecMLP_bus_{bus_idx}",
                  "bus_global_idx": bus_idx, "bus_name": buses_df.iloc[bus_idx]["Name"]})
        rows.append(m)
    return pd.DataFrame(rows)

def evaluate_baseline(pred, true, split_name):
    pred_3d = ensure_3d_output(pred, N_GEN, HOR)
    true_3d = ensure_3d_output(true, N_GEN, HOR)
    rows = []
    m = mae_rmse_nmae(true_3d, pred_3d)
    m.update({"split": split_name, "model": "Persistence_global"})
    rows.append(m)
    for j, bus_idx in enumerate(CONFIG["GENERATOR_BUS_IDX"]):
        m = mae_rmse_nmae(true_3d[:, j, :], pred_3d[:, j, :])
        m.update({"split": split_name, "model": f"Persistence_bus_{bus_idx}",
                  "bus_global_idx": bus_idx, "bus_name": buses_df.iloc[bus_idx]["Name"]})
        rows.append(m)
    return pd.DataFrame(rows)

metrics_df = pd.concat([
    evaluate_output_dict(train_out, "train"),
    evaluate_output_dict(val_out,   "val"),
    evaluate_output_dict(test_out,  "test"),
    evaluate_baseline(pers_train_pred, pers_train_true, "train"),
    evaluate_baseline(pers_val_pred,   pers_val_true,   "val"),
    evaluate_baseline(pers_test_pred,  pers_test_true,  "test"),
], ignore_index=True)

display(metrics_df)
print("\nRésumé test (global) :")
display(
    metrics_df[
        (metrics_df["split"] == "test") &
        (metrics_df["model"].str.contains("global"))
    ]
)


In [ ]:
# Cellule 26 — Diagnostic d'alignement
# MODIFIÉE : plus de batch.target_start_idx PyG — les metas sont déjà dans test_out["metas"].
def diagnose_alignment(out, n_checks=5):
    preds_3d = ensure_3d_output(out["preds_real"], N_GEN, HOR)
    trues_3d = ensure_3d_output(out["trues_real"], N_GEN, HOR)
    metas    = np.asarray(out["metas"]).reshape(-1)
    n = min(n_checks, len(metas), len(trues_3d))
    for k in range(n):
        ts0      = int(metas[k])
        expected = p_bus_generator[ts0 : ts0 + HOR].T
        assert np.allclose(expected, trues_3d[k], atol=1e-6), f"Erreur alignement k={k}"
        print(f"k={k} | target_start={ts0} | {ts_p[ts0]} → {ts_p[ts0+HOR-1]} | OK")

diagnose_alignment(test_out, n_checks=CONFIG["NUM_ALIGNMENT_CHECKS"])
print("Diagnostic d'alignement test OK.")


In [ ]:
# Cellule 27 — Métriques par horizon
# IDENTIQUE au notebook GNN (labels mis à jour).
def per_horizon_metrics(y_true, y_pred):
    y_true_3d = ensure_3d_output(y_true, N_GEN, HOR)
    y_pred_3d = ensure_3d_output(y_pred, N_GEN, HOR)
    rows = []
    for h in range(HOR):
        m = mae_rmse_nmae(y_true_3d[:, :, h], y_pred_3d[:, :, h])
        rows.append({"h": h + 1, **m})
    return pd.DataFrame(rows)

test_horizon_df      = per_horizon_metrics(test_out["trues_real"], test_out["preds_real"])
pers_test_horizon_df = per_horizon_metrics(pers_test_true, pers_test_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(test_horizon_df["h"], test_horizon_df["RMSE"], label="ElecMLP")
ax1.plot(pers_test_horizon_df["h"], pers_test_horizon_df["RMSE"], label="Persistence", linestyle="--")
ax1.set(xlabel="Horizon (h)", ylabel="RMSE", title="RMSE par horizon - test")
ax1.legend(); ax1.grid(True)

ax2.plot(test_horizon_df["h"], test_horizon_df["MAE"], label="ElecMLP")
ax2.plot(pers_test_horizon_df["h"], pers_test_horizon_df["MAE"], label="Persistence", linestyle="--")
ax2.set(xlabel="Horizon (h)", ylabel="MAE", title="MAE par horizon - test")
ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()
display(test_horizon_df.head())


In [ ]:
# Cellule 28 — Reconstruction continue sur le test
# IDENTIQUE au notebook GNN.
true_cont = np.full((T, N_GEN), np.nan)
pred_cont = np.full((T, N_GEN), np.nan)

preds_3d = ensure_3d_output(test_out["preds_real"], N_GEN, HOR)
metas    = test_out["metas"]
assert len(metas) == len(preds_3d)

for k, ts0 in enumerate(metas):
    ts0 = int(ts0)
    for j in range(N_GEN):
        true_cont[ts0 : ts0 + HOR, j] = p_bus_generator[ts0 : ts0 + HOR, j]
        pred_cont[ts0 : ts0 + HOR, j] = preds_3d[k, j, :]

n_valid = int(np.sum(np.any(~np.isnan(true_cont), axis=1)))
print(f"Reconstruction continue : {n_valid} heures valides sur {T}")


In [ ]:
# Cellule 29 — Zoom 72h par bus générateur
# IDENTIQUE au notebook GNN.
N_ZOOM = 72
valid_hours_mask = np.any(~np.isnan(true_cont), axis=1)
valid_idx        = np.where(valid_hours_mask)[0]
start_plot       = valid_idx[0]
end_plot         = min(start_plot + N_ZOOM, T)

print("Zoom 72h de", ts_p[start_plot], "à", ts_p[end_plot - 1])

for j in range(min(4, N_GEN)):
    bus_idx  = CONFIG["GENERATOR_BUS_IDX"][j]
    bus_name = buses_df.iloc[bus_idx]["Name"]
    plt.figure(figsize=(14, 4))
    plt.plot(ts_p[start_plot:end_plot], true_cont[start_plot:end_plot, j],
             label=f"True bus {bus_idx+1} ({bus_name})")
    plt.plot(ts_p[start_plot:end_plot], pred_cont[start_plot:end_plot, j],
             label=f"Pred bus {bus_idx+1} ({bus_name})")
    plt.title(f"Zoom 72h — ElecMLP — bus {bus_idx+1} {bus_name}")
    plt.xlabel("Time"); plt.ylabel("p (MW)"); plt.grid(True); plt.legend()
    plt.show()


In [ ]:
# Cellule 30 — Total agrégé zoom 72h
# IDENTIQUE au notebook GNN.
true_total_72h = np.nansum(true_cont[start_plot:end_plot, :], axis=1)
pred_total_72h = np.nansum(pred_cont[start_plot:end_plot, :], axis=1)

plt.figure(figsize=(14, 4))
plt.plot(ts_p[start_plot:end_plot], true_total_72h, label="True total 15 bus")
plt.plot(ts_p[start_plot:end_plot], pred_total_72h, label="Pred total 15 bus")
plt.title("Zoom 72h reconstruction continue — ElecMLP — total des 15 bus générateurs")
plt.xlabel("Time"); plt.ylabel("Total p (MW)"); plt.grid(True); plt.legend()
plt.show()


In [ ]:
# Cellule 31 — Métriques agrégées sur les totaux
# IDENTIQUE au notebook GNN (label mis à jour).
def aggregated_total_metrics(true_arr, pred_arr, n_gen, horizon):
    true_3d = ensure_3d_output(true_arr, n_gen, horizon)
    pred_3d = ensure_3d_output(pred_arr, n_gen, horizon)
    true_total = true_3d.sum(axis=1)
    pred_total = pred_3d.sum(axis=1)
    err_total  = pred_total - true_total
    mae_total  = np.mean(np.abs(err_total))
    rmse_total = np.sqrt(np.mean(err_total ** 2))
    mean_abs   = np.mean(np.abs(true_total))
    return {
        "MAE_total_agg" : mae_total,
        "RMSE_total_agg": rmse_total,
        "STD_error_agg" : np.std(err_total),
        "mean_abs_true" : mean_abs,
        "NMAE_total_agg": mae_total / (mean_abs + 1e-12),
    }, true_total, pred_total, err_total

mlp_metrics,  mlp_true_tot,  mlp_pred_tot,  _ = aggregated_total_metrics(
    test_out["trues_real"], test_out["preds_real"], N_GEN, HOR)
pers_metrics, pers_true_tot, pers_pred_tot, _ = aggregated_total_metrics(
    pers_test_true, pers_test_pred, N_GEN, HOR)

df_total_metrics = pd.DataFrame([
    {"model": "ElecMLP",     **mlp_metrics},
    {"model": "Persistence", **pers_metrics},
])
display(df_total_metrics)


In [ ]:
# Cellule 32 — Métriques globales validation + figure
# MODIFIÉE : label "ElecMLP".
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true).reshape(-1) - np.asarray(y_pred).reshape(-1)) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true).reshape(-1) - np.asarray(y_pred).reshape(-1))))

def roughness(x):
    x = np.asarray(x).reshape(-1)
    return float(np.mean(np.abs(np.diff(x)))) if len(x) >= 2 else 0.0

pred_real_3d = ensure_3d_output(val_out["preds_real"], N_GEN, HOR)
true_real_3d = ensure_3d_output(val_out["trues_real"], N_GEN, HOR)
p_pred_total = pred_real_3d.sum(axis=1).reshape(-1)
p_true_total = true_real_3d.sum(axis=1).reshape(-1)

metric_mae      = mae(p_true_total, p_pred_total)
metric_rmse     = rmse(p_true_total, p_pred_total)
rough_true      = roughness(p_true_total)
rough_pred      = roughness(p_pred_total)

metrics_val = {
    "MAE"          : metric_mae,
    "RMSE"         : metric_rmse,
    "MAE/mean"     : metric_mae / (np.mean(np.abs(p_true_total)) + 1e-12),
    "Roughness err": abs(rough_pred - rough_true),
}

print("=== ElecMLP | Global metrics on validation (total des 15 bus générateurs) ===")
for k, v in metrics_val.items():
    print(f"{k:<15}: {v:.6f}")
print(f"{'Roughness true':<15}: {rough_true:.6f}")
print(f"{'Roughness pred':<15}: {rough_pred:.6f}")

names, values = list(metrics_val.keys()), list(metrics_val.values())
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, values)
ax.set_title("ElecMLP — Global metrics on validation (p_total agrégé)")
ax.set_ylabel("Value"); ax.grid(True, axis="y", alpha=0.3)
ymax = max(values) if values else 1.0
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01 * ymax,
            f"{v:.4f}", ha="center", va="bottom", fontsize=10)
fig.tight_layout()

save_dir  = CONFIG["BASE_DIR"] / "figures"
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "global_metrics_val_elec_mlp.png"
fig.savefig(save_path, dpi=140, bbox_inches="tight")
plt.show()
print("Figure saved to:", save_path.resolve())


In [ ]:
# Cellule 33 — Analyse STD
# MODIFIÉE : label "ElecMLP".
pred_3d = ensure_3d_output(val_out["preds_real"], N_GEN, HOR)
true_3d = ensure_3d_output(val_out["trues_real"], N_GEN, HOR)

std_true_global       = float(np.std(true_3d))
std_pred_global       = float(np.std(pred_3d))
true_total_v          = true_3d.sum(axis=1)
pred_total_v          = pred_3d.sum(axis=1)
std_true_total_global = float(np.std(true_total_v))
std_pred_total_global = float(np.std(pred_total_v))

std_true_h       = np.std(true_3d, axis=(0, 1))
std_pred_h       = np.std(pred_3d, axis=(0, 1))
std_true_total_h = np.std(true_total_v, axis=0)
std_pred_total_h = np.std(pred_total_v, axis=0)
std_true_gen     = np.std(true_3d, axis=(0, 2))
std_pred_gen     = np.std(pred_3d, axis=(0, 2))
std_diff_gen_h   = np.std(pred_3d, axis=0) - np.std(true_3d, axis=0)

print(f"STD globale true : {std_true_global:.4f}  pred : {std_pred_global:.4f}")
print(f"STD totale agrégée true : {std_true_total_global:.4f}  pred : {std_pred_total_global:.4f}")

gen_diff = std_pred_gen - std_true_gen
for label, idxs in [("sous-variés", np.argsort(gen_diff)[:5]),
                     ("sur-variés",  np.argsort(gen_diff)[-5:])]:
    print(f"\nGénérateurs les plus {label}:")
    for idx in idxs:
        print(f"  Gen {idx:02d} | std_true={std_true_gen[idx]:.4f} | std_pred={std_pred_gen[idx]:.4f} | diff={gen_diff[idx]:.4f}")

fig = plt.figure(figsize=(16, 12))
w = 0.35

ax1 = plt.subplot(2, 2, 1)
x = np.arange(2)
ax1.bar(x - w/2, [std_true_global, std_true_total_global], width=w, label="True")
ax1.bar(x + w/2, [std_pred_global, std_pred_total_global], width=w, label="Pred")
ax1.set_xticks(x); ax1.set_xticklabels(["Global all", "Global total"])
ax1.set_title("STD globale"); ax1.set_ylabel("STD"); ax1.grid(True, axis="y", alpha=0.3); ax1.legend()

ax2 = plt.subplot(2, 2, 2)
h = np.arange(1, HOR + 1)
ax2.plot(h, std_true_h, label="True - all"); ax2.plot(h, std_pred_h, label="Pred - all")
ax2.plot(h, std_true_total_h, linestyle="--", label="True - total agg")
ax2.plot(h, std_pred_total_h, linestyle="--", label="Pred - total agg")
ax2.set_title("STD par horizon"); ax2.grid(True, alpha=0.3); ax2.legend()

ax3 = plt.subplot(2, 2, 3)
g = np.arange(N_GEN)
ax3.bar(g - w/2, std_true_gen, width=w, label="True")
ax3.bar(g + w/2, std_pred_gen, width=w, label="Pred")
ax3.set_title("STD par générateur"); ax3.grid(True, axis="y", alpha=0.3); ax3.legend()

ax4 = plt.subplot(2, 2, 4)
im = ax4.imshow(std_diff_gen_h, aspect="auto", origin="lower")
ax4.set_title("Heatmap: STD(pred) - STD(true)")
ax4.set_xlabel("Horizon"); ax4.set_ylabel("Generator index")
ax4.set_xticks(np.arange(HOR)); ax4.set_xticklabels(np.arange(1, HOR + 1))
fig.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

fig.suptitle("STD analysis — ElecMLP (validation)", fontsize=14)
fig.tight_layout(); plt.show()


In [ ]:
# Cellule 34 — Aliases diagnostics
# IDENTIQUE au notebook GNN.
ts_                = ts_p
dyn_tensor_        = dyn_tensor
dyn_feature_names_ = dyn_feature_names
p_target_          = p_bus_generator
GEN_IDX_           = CONFIG["GENERATOR_BUS_IDX"]
N_GEN_             = N_GEN
HOR_               = HOR
test_out_          = test_out
pers_test_pred_    = pers_test_pred
pers_test_true_    = pers_test_true
buses_df_          = buses_df

def _to_3d_pred(arr, n_gen, hor):
    return ensure_3d_output(arr, n_gen, hor)

def mae_rmse(y_true, y_pred):
    y_true_f = np.asarray(y_true).reshape(-1)
    y_pred_f = np.asarray(y_pred).reshape(-1)
    return {"MAE": float(np.mean(np.abs(y_true_f - y_pred_f))),
            "RMSE": float(np.sqrt(np.mean((y_true_f - y_pred_f)**2)))}

print("Aliases diagnostics définis (ts_, dyn_tensor_, p_target_, test_out_, etc.)")


In [ ]:
# Cellule 35 — Distribution des features et corrélations
# IDENTIQUE au notebook GNN.
n_feat_show = min(6, dyn_tensor_.shape[-1])
feat_names  = dyn_feature_names_[:n_feat_show]

fig, axes = plt.subplots(2, n_feat_show, figsize=(4 * n_feat_show, 8))
for k in range(n_feat_show):
    vals = dyn_tensor_[..., k].reshape(-1)
    vals = vals[np.isfinite(vals)]
    axes[0, k].hist(vals, bins=60)
    axes[0, k].set_title(f"Hist {feat_names[k]}")
    axes[0, k].grid(True)
    sample = vals[np.linspace(0, len(vals)-1, min(4000, len(vals))).astype(int)]
    axes[1, k].boxplot(sample, vert=True)
    axes[1, k].set_title(f"Box {feat_names[k]}")
    axes[1, k].grid(True)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_vals = p_target_.reshape(-1)
target_vals = target_vals[np.isfinite(target_vals)]
axes[0].hist(target_vals, bins=80)
axes[0].set_title("Distribution globale des cibles"); axes[0].grid(True)
axes[1].boxplot(target_vals[np.linspace(0, len(target_vals)-1, min(4000, len(target_vals))).astype(int)], vert=True)
axes[1].set_title("Boxplot global des cibles"); axes[1].grid(True)
plt.tight_layout(); plt.show()

corr_dict = {}
for i in range(n_feat_show):
    corr_dict[f"{feat_names[i]}_mean"] = np.nanmean(dyn_tensor_[:, :, i], axis=1)
for j in range(min(6, p_target_.shape[1])):
    corr_dict[f"target_bus_{GEN_IDX_[j]}"] = p_target_[:, j]

corr_df  = pd.DataFrame(corr_dict)
corr_mat = corr_df.corr(numeric_only=True).values
plt.figure(figsize=(10, 8))
plt.imshow(corr_mat, aspect="auto")
plt.colorbar(label="corr")
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90)
plt.yticks(range(len(corr_df.columns)), corr_df.columns)
plt.title("Heatmap de corrélations (features agrégées + cibles) — ElecMLP")
plt.tight_layout(); plt.show()


In [ ]:
# Cellule 36 — RMSE/MAE par horizon + profils 24h
# IDENTIQUE au notebook GNN.
y_true3 = _to_3d_pred(test_out_["trues_real"], N_GEN_, HOR_)
y_pred3 = _to_3d_pred(test_out_["preds_real"], N_GEN_, HOR_)
b_true3 = _to_3d_pred(pers_test_true_,         N_GEN_, HOR_)
b_pred3 = _to_3d_pred(pers_test_pred_,         N_GEN_, HOR_)

rows = []
for h in range(HOR_):
    m = mae_rmse(y_true3[:, :, h], y_pred3[:, :, h])
    b = mae_rmse(b_true3[:, :, h], b_pred3[:, :, h])
    rows.append({
        "horizon_h"              : h + 1,
        "model_MAE"              : m["MAE"],
        "model_RMSE"             : m["RMSE"],
        "baseline_MAE"           : b["MAE"],
        "baseline_RMSE"          : b["RMSE"],
        "RMSE_gain_vs_baseline_%": 100 * (b["RMSE"] - m["RMSE"]) / (b["RMSE"] + 1e-12),
    })
horizon_df = pd.DataFrame(rows)
display(horizon_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(horizon_df["horizon_h"], horizon_df["model_RMSE"],    label="ElecMLP")
axes[0].plot(horizon_df["horizon_h"], horizon_df["baseline_RMSE"], label="baseline")
axes[0].set_title("RMSE par horizon"); axes[0].grid(True); axes[0].legend()

axes[1].plot(horizon_df["horizon_h"], horizon_df["model_MAE"],    label="ElecMLP")
axes[1].plot(horizon_df["horizon_h"], horizon_df["baseline_MAE"], label="baseline")
axes[1].set_title("MAE par horizon"); axes[1].grid(True); axes[1].legend()
plt.tight_layout(); plt.show()

sel = np.linspace(0, len(y_true3)-1, min(3, len(y_true3))).astype(int)
for k in sel:
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(1, HOR_+1), np.sum(y_true3[k], axis=0), label="true total")
    plt.plot(np.arange(1, HOR_+1), np.sum(y_pred3[k], axis=0), label="pred total")
    plt.plot(np.arange(1, HOR_+1), np.sum(b_pred3[k], axis=0), label="baseline total")
    plt.title(f"Profil 24h | fenêtre test {k} — ElecMLP"); plt.grid(True); plt.legend(); plt.show()


In [ ]:
# Cellule 37 — Métriques par bus générateur
# IDENTIQUE au notebook GNN (label mis à jour).
y_true3 = _to_3d_pred(test_out_["trues_real"], N_GEN_, HOR_)
y_pred3 = _to_3d_pred(test_out_["preds_real"], N_GEN_, HOR_)

rows = []
for j in range(N_GEN_):
    yt = y_true3[:, j, :]
    yp = y_pred3[:, j, :]
    m  = mae_rmse(yt, yp)
    rows.append({
        "gen_bus_idx"      : GEN_IDX_[j],
        "MAE"              : m["MAE"],
        "RMSE"             : m["RMSE"],
        "mean_abs_target"  : float(np.mean(np.abs(yt))),
        "NMAE"             : float(m["MAE"] / (np.mean(np.abs(yt)) + 1e-12)),
        "std_true"         : float(np.std(yt)),
        "std_pred"         : float(np.std(yp)),
        "std_ratio_pred_true": float(np.std(yp) / (np.std(yt) + 1e-12)),
    })
node_df = pd.DataFrame(rows).sort_values("RMSE", ascending=False)
display(node_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(node_df["gen_bus_idx"].astype(str), node_df["RMSE"])
axes[0].set_title("RMSE par bus générateur — ElecMLP")
axes[0].tick_params(axis="x", rotation=90); axes[0].grid(True, axis="y")

axes[1].bar(node_df["gen_bus_idx"].astype(str), node_df["std_ratio_pred_true"])
axes[1].axhline(1.0, linestyle="--")
axes[1].set_title("std(pred) / std(true) par bus — ElecMLP")
axes[1].tick_params(axis="x", rotation=90); axes[1].grid(True, axis="y")
plt.tight_layout(); plt.show()
print("Pires variables cibles (bus) :")
display(node_df.head(10))


In [ ]:
# Cellule 38 — Analyse des erreurs extrêmes
# IDENTIQUE au notebook GNN.
y_true3 = _to_3d_pred(test_out_["trues_real"], N_GEN_, HOR_)
y_pred3 = _to_3d_pred(test_out_["preds_real"], N_GEN_, HOR_)
abs_err = np.abs(y_pred3 - y_true3)

thr          = np.nanpercentile(np.abs(y_true3), 95)
mask_extreme = np.abs(y_true3) >= thr
mask_normal  = np.abs(y_true3) <  thr

extreme_mae = float(np.mean(abs_err[mask_extreme])) if np.any(mask_extreme) else np.nan
normal_mae  = float(np.mean(abs_err[mask_normal]))  if np.any(mask_normal)  else np.nan

display(pd.DataFrame([{
    "threshold_abs_y_95pct"   : thr,
    "MAE_normal_values"       : normal_mae,
    "MAE_extreme_values"      : extreme_mae,
    "extreme_over_normal_ratio": extreme_mae / (normal_mae + 1e-12),
}]))

n_win, n_gen, n_hor = y_true3.shape
flat_idx = np.argsort(abs_err.reshape(-1))[-20:]
coords   = np.array(np.unravel_index(flat_idx, (n_win, n_gen, n_hor))).T
rows = []
for w, j, h in coords[::-1]:
    rows.append({"window_idx": int(w), "bus_idx": int(GEN_IDX_[j]),
                 "horizon_h": int(h+1), "y_true": float(y_true3[w,j,h]),
                 "y_pred": float(y_pred3[w,j,h]), "abs_error": float(abs_err[w,j,h])})
top_err_df = pd.DataFrame(rows)
display(top_err_df)

for _, r in top_err_df.head(3).iterrows():
    w = int(r["window_idx"])
    j = list(GEN_IDX_).index(int(r["bus_idx"]))
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(1, HOR_+1), y_true3[w, j], label="true")
    plt.plot(np.arange(1, HOR_+1), y_pred3[w, j], label="pred")
    plt.axvline(int(r["horizon_h"]), linestyle="--", label="erreur max")
    plt.title(f"Extrême | fenêtre={w} | bus={GEN_IDX_[j]} | err={r['abs_error']:.3f}")
    plt.grid(True); plt.legend(); plt.show()


In [ ]:
# Cellule 39 — Style matplotlib
# IDENTIQUE au notebook GNN.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

plt.rcParams.update({
    "figure.figsize": (14, 5), "axes.titlesize": 17, "axes.labelsize": 14,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
    "figure.titlesize": 19, "lines.linewidth": 2.0, "axes.grid": True,
    "grid.alpha": 0.35, "axes.spines.top": False, "axes.spines.right": False,
})

def _get(*names, default=None):
    for n in names:
        if n in globals(): return globals()[n]
    return default


In [ ]:
# Cellule 40 — Comparaison ElecMLP vs Persistence sur les totaux agrégés
# ADAPTÉE : dans le notebook GNN cette cellule comparait GNN vs NoMessagePassing vs Persistence.
# Ici, le modèle principal est déjà un MLP : pas de deuxième modèle à comparer.
# On compare ElecMLP vs Persistence uniquement.

def aggregate_totals_per_hour(arr, n_gen, horizon):
    return ensure_3d_output(arr, n_gen, horizon).sum(axis=1)

mlp_test_pred_tot  = aggregate_totals_per_hour(test_out["preds_real"],  N_GEN, HOR)
mlp_test_true_tot  = aggregate_totals_per_hour(test_out["trues_real"],  N_GEN, HOR)
pers_test_pred_tot = aggregate_totals_per_hour(pers_test_pred,          N_GEN, HOR)
pers_test_true_tot = aggregate_totals_per_hour(pers_test_true,          N_GEN, HOR)

rows = []
for label, true_tot, pred_tot in [
    ("ElecMLP_aggregated_total",     mlp_test_true_tot,  mlp_test_pred_tot),
    ("Persistence_aggregated_total", pers_test_true_tot, pers_test_pred_tot),
]:
    m = mae_rmse_nmae(true_tot, pred_tot)
    m["model"] = label
    rows.append(m)

agg_total_df = pd.DataFrame(rows)[["model", "MAE", "RMSE", "NMAE"]].sort_values("RMSE").reset_index(drop=True)
print("=== Comparaison TEST sur les TOTAUX AGRÉGÉS ===")
display(agg_total_df)


In [ ]:
# Cellule 41 — Roughness sur les totaux agrégés
# ADAPTÉE : label "ElecMLP".
def roughness(x):
    x = np.asarray(x).reshape(-1)
    return float(np.mean(np.abs(np.diff(x)))) if len(x) >= 2 else 0.0

rows = []
for label, true_tot, pred_tot in [
    ("ElecMLP",    mlp_test_true_tot,  mlp_test_pred_tot),
    ("Persistence",pers_test_true_tot, pers_test_pred_tot),
]:
    rows.append({
        "model"      : label,
        "rough_true" : roughness(true_tot),
        "rough_pred" : roughness(pred_tot),
        "rough_error": roughness(pred_tot - true_tot),
    })

roughness_df = pd.DataFrame(rows)
roughness_df["pred_minus_true"] = roughness_df["rough_pred"] - roughness_df["rough_true"]
roughness_df["pred_over_true"]  = roughness_df["rough_pred"] / np.maximum(roughness_df["rough_true"], 1e-12)
print("=== Roughness sur les TOTAUX AGRÉGÉS (TEST) ===")
display(roughness_df)

print("\n=== Lecture rapide ===")
for _, r in roughness_df.iterrows():
    print(f"{r['model']}:")
    print(f"  rough_true={r['rough_true']:.4f}  rough_pred={r['rough_pred']:.4f}  pred/true={r['pred_over_true']:.3f}")
